# Vibe-Aware Vocabulary Pruning: Benchmarking a Dual-Encoder Router for LLM Inference

This notebook implements and benchmarks four vocabulary routing strategies against a full-vocabulary baseline:

| Section | Method |
|---------|--------|
| 1 | Baseline profiling (full vocab) |
| 2 | Vocab space exploration (PCA/UMAP) |
| 3 | Static Top-K + Cosine router |
| 4 | Cluster-based router (k-means) |
| 5 | **Dual-Encoder router (MNRL)** — main contribution |
| 6 | Pareto charts: accuracy vs. speed |
| 7 | Discussion |

All pruned-vocabulary metrics are reported **relative to the full-vocabulary baseline**.

## 0. Setup, Imports, Hardware Info

In [ ]:
import time
import math
import random
import warnings
from pathlib import Path
from typing import List, Tuple, Dict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from sklearn.cluster import MiniBatchKMeans
from sklearn.decomposition import PCA

warnings.filterwarnings('ignore')
sns.set_theme(style='darkgrid', palette='muted')

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Device selection: MPS > CUDA > CPU
if torch.backends.mps.is_available():
    DEVICE = torch.device('mps')
elif torch.cuda.is_available():
    DEVICE = torch.device('cuda')
else:
    DEVICE = torch.device('cpu')

print(f'Device: {DEVICE}')
print(f'PyTorch: {torch.__version__}')
if DEVICE.type == 'mps':
    print('Backend: Apple MPS')
elif DEVICE.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# ── Configuration ────────────────────────────────────────────────────────────
MODEL_NAME = 'meta-llama/Llama-3.2-1B-Instruct'
# Fallback if Llama requires gating:
# MODEL_NAME = 'microsoft/Phi-3-mini-4k-instruct'

DATASET_NAME = 'wikitext'
DATASET_CONFIG = 'wikitext-2-raw-v1'

K_VALUES = [512, 1_000, 2_000, 5_000, 10_000]  # shortlist sizes to sweep
N_BENCH_PROMPTS = 50          # prompts for latency benchmarks
BENCH_MAX_NEW_TOKENS = 64     # tokens generated per prompt in latency runs
PPL_MAX_TOKENS = 4096         # tokens used for perplexity evaluation
ROUTER_TRAIN_EPOCHS = 3
ROUTER_BATCH_SIZE = 256
ROUTER_LR = 3e-4
ROUTER_HIDDEN_DIM = 512       # MLP hidden dim
ROUTER_OUT_DIM = 256          # projected query dim
TEMPERATURE = 0.05            # InfoNCE temperature
PREFIX_FRAC_LOW = 0.4
PREFIX_FRAC_HIGH = 0.7
N_KMEANS_CLUSTERS = 256       # for cluster router

RESULTS_DIR = Path('results')
RESULTS_DIR.mkdir(exist_ok=True)

## 0.1 Load Model and Tokenizer

In [ ]:
print(f'Loading tokenizer and model: {MODEL_NAME}')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    device_map='auto' if DEVICE.type != 'mps' else None,
)
if DEVICE.type == 'mps':
    model = model.to(DEVICE)
model.eval()

VOCAB_SIZE = model.config.vocab_size
HIDDEN_DIM = model.config.hidden_size
print(f'Vocab size: {VOCAB_SIZE:,}')
print(f'Hidden dim: {HIDDEN_DIM}')
print(f'Model dtype: {next(model.parameters()).dtype}')

# Extract the vocabulary embedding matrix (lm_head weights) as float32 on CPU
# Shape: [vocab_size, hidden_dim]
with torch.no_grad():
    lm_head_weight = model.lm_head.weight.detach().float().cpu()
print(f'lm_head weight shape: {lm_head_weight.shape}')

# Pre-normalised vocabulary embeddings for cosine lookups
lm_head_norm = F.normalize(lm_head_weight, dim=-1)  # [V, d]

## 0.2 Load Dataset

In [ ]:
raw_dataset = load_dataset(DATASET_NAME, DATASET_CONFIG)
print(raw_dataset)

def get_text_chunks(split: str, chunk_tokens: int = 128, n_chunks: int = None) -> List[torch.Tensor]:
    """Tokenise the split and return a list of fixed-length token tensors."""
    text = ' '.join(
        line for line in raw_dataset[split]['text'] if len(line.strip()) > 20
    )
    tokens = tokenizer.encode(text, return_tensors='pt')[0]
    chunks = [tokens[i:i+chunk_tokens] for i in range(0, len(tokens) - chunk_tokens, chunk_tokens)]
    if n_chunks is not None:
        chunks = chunks[:n_chunks]
    return chunks

bench_prompts = get_text_chunks('validation', chunk_tokens=128, n_chunks=N_BENCH_PROMPTS)
print(f'Benchmark prompts: {len(bench_prompts)} x 128 tokens')

---
## 1. Baseline Profiling

Measure:
- Overall tokens/sec with full-vocabulary decoding
- Fraction of per-token time consumed by the `lm_head` projection

In [ ]:
def timed_lm_head(hidden: torch.Tensor, weight: torch.Tensor) -> Tuple[torch.Tensor, float]:
    """Run lm_head projection and return (logits, elapsed_seconds)."""
    if DEVICE.type == 'mps':
        torch.mps.synchronize()
    elif DEVICE.type == 'cuda':
        torch.cuda.synchronize()
    t0 = time.perf_counter()
    logits = hidden @ weight.T
    if DEVICE.type == 'mps':
        torch.mps.synchronize()
    elif DEVICE.type == 'cuda':
        torch.cuda.synchronize()
    return logits, time.perf_counter() - t0


def baseline_decode_with_profile(
    input_ids: torch.Tensor,
    max_new_tokens: int = BENCH_MAX_NEW_TOKENS,
) -> Dict:
    """Greedy decode with full vocab; track per-step timings."""
    input_ids = input_ids.unsqueeze(0).to(DEVICE)
    past_key_values = None
    total_times, lmhead_times = [], []

    with torch.no_grad():
        for _ in range(max_new_tokens):
            t_start = time.perf_counter()

            out = model(
                input_ids=input_ids if past_key_values is None else input_ids[:, -1:],
                past_key_values=past_key_values,
                use_cache=True,
                output_hidden_states=True,
            )
            hidden = out.hidden_states[-1][:, -1, :]  # [1, d]
            past_key_values = out.past_key_values

            lm_w = model.lm_head.weight  # on device
            logits, lmhead_t = timed_lm_head(hidden, lm_w)

            next_token = logits[0].argmax(dim=-1, keepdim=True).unsqueeze(0)
            input_ids = next_token

            if DEVICE.type == 'mps':
                torch.mps.synchronize()
            t_end = time.perf_counter()

            total_times.append(t_end - t_start)
            lmhead_times.append(lmhead_t)

    return {
        'total_times': total_times,
        'lmhead_times': lmhead_times,
    }


print('Running baseline profiling (this may take a few minutes)...')
# Warm-up
_ = baseline_decode_with_profile(bench_prompts[0], max_new_tokens=5)

baseline_results = []
for prompt in tqdm(bench_prompts, desc='Baseline profiling'):
    r = baseline_decode_with_profile(prompt)
    baseline_results.append(r)

all_total = np.array([t for r in baseline_results for t in r['total_times']])
all_lmhead = np.array([t for r in baseline_results for t in r['lmhead_times']])

baseline_tps = 1.0 / all_total.mean()
lmhead_frac = all_lmhead.mean() / all_total.mean()

print(f'\n=== Baseline Results ===')
print(f'Tokens/sec:            {baseline_tps:.2f}')
print(f'Mean total time/token: {all_total.mean()*1000:.2f} ms')
print(f'Mean lm_head time:     {all_lmhead.mean()*1000:.2f} ms')
print(f'lm_head fraction:      {lmhead_frac*100:.1f}%')

# Save
baseline_summary = {
    'tokens_per_sec': baseline_tps,
    'mean_total_ms': all_total.mean() * 1000,
    'mean_lmhead_ms': all_lmhead.mean() * 1000,
    'lmhead_frac': lmhead_frac,
}
pd.DataFrame([baseline_summary]).to_csv(RESULTS_DIR / 'baseline_summary.csv', index=False)

In [ ]:
# Visualise baseline timing breakdown
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(all_total * 1000, bins=40, alpha=0.75, label='Total')
axes[0].hist(all_lmhead * 1000, bins=40, alpha=0.75, label='lm_head')
axes[0].set_xlabel('Time per token (ms)')
axes[0].set_ylabel('Count')
axes[0].set_title('Per-token time distribution')
axes[0].legend()

labels = ['lm_head', 'other']
sizes = [lmhead_frac, 1 - lmhead_frac]
axes[1].pie(sizes, labels=labels, autopct='%1.1f%%', startangle=90)
axes[1].set_title('Time breakdown per token')

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'baseline_profile.png', dpi=150)
plt.show()

---
## 2. Vocabulary Space Exploration

Visualise the token embedding space to understand cluster structure before designing routers.

In [ ]:
print('Reducing vocabulary embeddings with PCA...')
# Sample 8k tokens for visualisation speed
vis_idx = np.random.choice(VOCAB_SIZE, size=min(8000, VOCAB_SIZE), replace=False)
vis_emb = lm_head_weight[vis_idx].numpy()  # [8000, d]

pca = PCA(n_components=2, random_state=SEED)
vis_2d = pca.fit_transform(vis_emb)

fig, ax = plt.subplots(figsize=(8, 8))
ax.scatter(vis_2d[:, 0], vis_2d[:, 1], s=1, alpha=0.3)
ax.set_title(f'Token embedding PCA (n={len(vis_idx):,} sampled tokens)')
ax.set_xlabel('PC1')
ax.set_ylabel('PC2')
explained = pca.explained_variance_ratio_.sum() * 100
ax.text(0.02, 0.98, f'Explained variance: {explained:.1f}%',
        transform=ax.transAxes, va='top')
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'vocab_pca.png', dpi=150)
plt.show()

In [ ]:
# Optional UMAP (skip if umap-learn not installed)
try:
    import umap
    print('Reducing with UMAP (this takes ~1-2 min)...')
    reducer = umap.UMAP(n_components=2, n_neighbors=30, min_dist=0.1,
                        random_state=SEED, verbose=False)
    vis_umap = reducer.fit_transform(vis_emb)
    fig, ax = plt.subplots(figsize=(8, 8))
    ax.scatter(vis_umap[:, 0], vis_umap[:, 1], s=1, alpha=0.3)
    ax.set_title(f'Token embedding UMAP (n={len(vis_idx):,})')
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / 'vocab_umap.png', dpi=150)
    plt.show()
except ImportError:
    print('umap-learn not installed, skipping UMAP visualisation.')

---
## 3. Static and Cosine Router Baselines

- **Static Top-K:** always keep the K tokens with the highest `lm_head` weight L2-norm (context-free).
- **Cosine router:** at each step, keep the K tokens with highest cosine similarity to the current hidden state.

In [ ]:
# Pre-compute static top-K indices (sorted by weight norm, descending)
token_norms = lm_head_weight.norm(dim=-1)  # [V]
static_sorted_idx = token_norms.argsort(descending=True)  # [V]

def get_static_shortlist(k: int) -> torch.Tensor:
    return static_sorted_idx[:k]

def get_cosine_shortlist(hidden: torch.Tensor, k: int) -> torch.Tensor:
    """hidden: [d] float32 on CPU."""
    h_norm = F.normalize(hidden.unsqueeze(0), dim=-1)  # [1, d]
    sims = (h_norm @ lm_head_norm.T).squeeze(0)        # [V]
    return sims.topk(k).indices                         # [k]

In [ ]:
def evaluate_router(
    router_fn,           # callable(hidden_cpu, k) -> LongTensor of k indices
    k: int,
    n_prompts: int = N_BENCH_PROMPTS,
    max_new_tokens: int = BENCH_MAX_NEW_TOKENS,
) -> Dict:
    """
    Run greedy decode with a pruned vocabulary.
    Returns tokens/sec, acceptance rate (vs full-vocab greedy), lmhead_frac.
    """
    accepted, total_steps = 0, 0
    total_times, lmhead_times = [], []

    for prompt in tqdm(bench_prompts[:n_prompts], desc=f'K={k}', leave=False):
        input_ids = prompt.unsqueeze(0).to(DEVICE)
        past_key_values = None

        with torch.no_grad():
            for _ in range(max_new_tokens):
                t_start = time.perf_counter()

                out = model(
                    input_ids=input_ids if past_key_values is None else input_ids[:, -1:],
                    past_key_values=past_key_values,
                    use_cache=True,
                    output_hidden_states=True,
                )
                hidden = out.hidden_states[-1][:, -1, :]  # [1, d] on device
                past_key_values = out.past_key_values

                # Full-vocab greedy token (reference)
                full_logits = hidden @ model.lm_head.weight.T  # [1, V]
                gold_token = full_logits[0].argmax().item()

                # Router shortlist
                hidden_cpu = hidden[0].float().cpu()
                shortlist = router_fn(hidden_cpu, k)  # [k]

                # Pruned lm_head
                pruned_weight = model.lm_head.weight[shortlist]  # [k, d]
                t_lm0 = time.perf_counter()
                pruned_logits = hidden @ pruned_weight.T  # [1, k]
                if DEVICE.type == 'mps':
                    torch.mps.synchronize()
                t_lm1 = time.perf_counter()

                # Map pruned argmax back to vocab
                pruned_best_local = pruned_logits[0].argmax().item()
                pruned_best_token = shortlist[pruned_best_local].item()

                accepted += int(pruned_best_token == gold_token)
                total_steps += 1

                # Next input: use gold token to keep decode on track
                input_ids = torch.tensor([[gold_token]], device=DEVICE)

                if DEVICE.type == 'mps':
                    torch.mps.synchronize()
                t_end = time.perf_counter()

                total_times.append(t_end - t_start)
                lmhead_times.append(t_lm1 - t_lm0)

    arr_total = np.array(total_times)
    arr_lm = np.array(lmhead_times)
    return {
        'k': k,
        'acceptance_rate': accepted / total_steps,
        'tokens_per_sec': 1.0 / arr_total.mean(),
        'lmhead_frac': arr_lm.mean() / arr_total.mean(),
        'mean_total_ms': arr_total.mean() * 1000,
        'mean_lmhead_ms': arr_lm.mean() * 1000,
    }


def compute_perplexity(
    router_fn,
    k: int,
    max_tokens: int = PPL_MAX_TOKENS,
) -> float:
    """
    Compute perplexity on WikiText-2 test split using pruned logits.
    Returns perplexity value.
    """
    test_text = ' '.join(
        line for line in raw_dataset['test']['text'] if len(line.strip()) > 20
    )
    tokens = tokenizer.encode(test_text, return_tensors='pt')[0][:max_tokens]
    input_ids = tokens.unsqueeze(0).to(DEVICE)

    stride = 512
    seq_len = input_ids.size(1)
    nlls = []

    with torch.no_grad():
        for begin in tqdm(range(0, seq_len - 1, stride), desc=f'PPL K={k}', leave=False):
            end = min(begin + stride, seq_len - 1)
            chunk = input_ids[:, begin:end + 1]
            out = model(
                input_ids=chunk,
                output_hidden_states=True,
            )
            hiddens = out.hidden_states[-1][:, :-1, :]  # [1, T-1, d]
            targets = chunk[:, 1:]                      # [1, T-1]

            for t in range(hiddens.size(1)):
                h = hiddens[0, t].float().cpu()         # [d]
                shortlist = router_fn(h, k)             # [k]
                pruned_w = model.lm_head.weight[shortlist].float().cpu()
                logits = h @ pruned_w.T                 # [k]
                target = targets[0, t].item()

                # If gold token in shortlist, compute true NLL; else assign max NLL
                if target in shortlist.tolist():
                    local_idx = (shortlist == target).nonzero(as_tuple=True)[0][0].item()
                    nll = -F.log_softmax(logits, dim=-1)[local_idx].item()
                else:
                    nll = -math.log(1e-9)  # penalise blind spots heavily
                nlls.append(nll)

    return math.exp(np.mean(nlls))

In [ ]:
# Compute full-vocab perplexity baseline
def full_vocab_ppl(max_tokens: int = PPL_MAX_TOKENS) -> float:
    test_text = ' '.join(
        line for line in raw_dataset['test']['text'] if len(line.strip()) > 20
    )
    tokens = tokenizer.encode(test_text, return_tensors='pt')[0][:max_tokens]
    input_ids = tokens.unsqueeze(0).to(DEVICE)
    stride = 512
    seq_len = input_ids.size(1)
    nlls = []
    with torch.no_grad():
        for begin in tqdm(range(0, seq_len - 1, stride), desc='PPL baseline', leave=False):
            end = min(begin + stride, seq_len - 1)
            chunk = input_ids[:, begin:end + 1]
            out = model(input_ids=chunk)
            logits = out.logits[0, :-1, :]   # [T-1, V]
            targets = chunk[0, 1:]            # [T-1]
            nll = F.cross_entropy(logits, targets, reduction='none')
            nlls.extend(nll.float().cpu().tolist())
    return math.exp(np.mean(nlls))

print('Computing full-vocab baseline perplexity...')
baseline_ppl = full_vocab_ppl()
print(f'Baseline perplexity: {baseline_ppl:.3f}')

In [ ]:
# Sweep K for Static and Cosine routers
static_results, cosine_results = [], []

for k in K_VALUES:
    print(f'\n--- Static router K={k} ---')
    static_shortlist_cached = get_static_shortlist(k)
    r = evaluate_router(lambda h, k, sl=static_shortlist_cached: sl, k=k)
    r['perplexity'] = compute_perplexity(lambda h, k, sl=static_shortlist_cached: sl, k=k)
    r['delta_ppl'] = r['perplexity'] - baseline_ppl
    static_results.append(r)
    print(f"  Acceptance: {r['acceptance_rate']:.3f}  PPL: {r['perplexity']:.3f}  ΔPPL: {r['delta_ppl']:+.3f}  TPS: {r['tokens_per_sec']:.2f}")

for k in K_VALUES:
    print(f'\n--- Cosine router K={k} ---')
    r = evaluate_router(get_cosine_shortlist, k=k)
    r['perplexity'] = compute_perplexity(get_cosine_shortlist, k=k)
    r['delta_ppl'] = r['perplexity'] - baseline_ppl
    cosine_results.append(r)
    print(f"  Acceptance: {r['acceptance_rate']:.3f}  PPL: {r['perplexity']:.3f}  ΔPPL: {r['delta_ppl']:+.3f}  TPS: {r['tokens_per_sec']:.2f}")

pd.DataFrame(static_results).to_csv(RESULTS_DIR / 'static_router_results.csv', index=False)
pd.DataFrame(cosine_results).to_csv(RESULTS_DIR / 'cosine_router_results.csv', index=False)

---
## 4. Cluster-Based Router

Offline k-means clustering of the vocabulary embedding space. At inference time, select the top-C clusters by cosine similarity to the hidden state, and return all tokens in those clusters as the shortlist.

In [ ]:
print(f'Fitting k-means with {N_KMEANS_CLUSTERS} clusters on vocabulary embeddings...')
kmeans = MiniBatchKMeans(
    n_clusters=N_KMEANS_CLUSTERS,
    random_state=SEED,
    batch_size=4096,
    n_init=3,
)
kmeans.fit(lm_head_weight.numpy())
cluster_labels = torch.tensor(kmeans.labels_, dtype=torch.long)  # [V]
cluster_centers = torch.tensor(kmeans.cluster_centers_, dtype=torch.float32)  # [C, d]
cluster_centers_norm = F.normalize(cluster_centers, dim=-1)

# Precompute per-cluster token lists
cluster_to_tokens: Dict[int, torch.Tensor] = {}
for c in range(N_KMEANS_CLUSTERS):
    cluster_to_tokens[c] = (cluster_labels == c).nonzero(as_tuple=True)[0]

cluster_sizes = [len(cluster_to_tokens[c]) for c in range(N_KMEANS_CLUSTERS)]
print(f'Cluster sizes: min={min(cluster_sizes)}, max={max(cluster_sizes)}, mean={np.mean(cluster_sizes):.0f}')


def get_cluster_shortlist(hidden: torch.Tensor, k: int) -> torch.Tensor:
    """Select clusters until we have at least k tokens."""
    h_norm = F.normalize(hidden.unsqueeze(0), dim=-1)  # [1, d]
    sims = (h_norm @ cluster_centers_norm.T).squeeze(0)  # [C]
    sorted_clusters = sims.argsort(descending=True)
    selected = []
    for c in sorted_clusters.tolist():
        selected.append(cluster_to_tokens[c])
        if sum(len(s) for s in selected) >= k:
            break
    shortlist = torch.cat(selected)[:k]
    return shortlist

In [ ]:
cluster_results = []
for k in K_VALUES:
    print(f'\n--- Cluster router K={k} ---')
    r = evaluate_router(get_cluster_shortlist, k=k)
    r['perplexity'] = compute_perplexity(get_cluster_shortlist, k=k)
    r['delta_ppl'] = r['perplexity'] - baseline_ppl
    cluster_results.append(r)
    print(f"  Acceptance: {r['acceptance_rate']:.3f}  PPL: {r['perplexity']:.3f}  ΔPPL: {r['delta_ppl']:+.3f}  TPS: {r['tokens_per_sec']:.2f}")

pd.DataFrame(cluster_results).to_csv(RESULTS_DIR / 'cluster_router_results.csv', index=False)

---
## 5. Dual-Encoder Router (Main Contribution)

Train a lightweight MLP head that maps the LLM's hidden state to a query vector in the token embedding space. At inference time, perform a top-K cosine search over the vocabulary to retrieve the shortlist.

### Architecture
- **Tower A:** frozen LLM hidden state → MLP projection head → query vector ∈ ℝ^d_r
- **Tower B:** token embedding rows from `lm_head` weight matrix (frozen)
- **Loss:** MNRL / InfoNCE with in-batch negatives

### 5a. Dataset Construction

In [ ]:
class PrefixHiddenStateDataset(Dataset):
    """
    For each training sentence, sample a prefix split point.
    Returns (hidden_state_of_last_prefix_token, gold_next_token_idx).
    Hidden states are pre-extracted to avoid re-running the LLM during training.
    """
    def __init__(self, split: str = 'train', max_sentences: int = 5000):
        self.samples: List[Tuple[torch.Tensor, int]] = []
        sentences = [
            line.strip() for line in raw_dataset[split]['text']
            if len(line.strip()) > 40
        ][:max_sentences]

        print(f'Extracting hidden states from {len(sentences)} sentences...')
        model.eval()
        with torch.no_grad():
            for sent in tqdm(sentences, desc='Building dataset'):
                ids = tokenizer.encode(sent, return_tensors='pt')[0]
                T = len(ids)
                if T < 8:
                    continue
                split_pt = random.randint(
                    int(PREFIX_FRAC_LOW * T),
                    int(PREFIX_FRAC_HIGH * T)
                )
                prefix = ids[:split_pt].unsqueeze(0).to(DEVICE)
                gold_token = ids[split_pt].item()

                out = model(input_ids=prefix, output_hidden_states=True)
                h = out.hidden_states[-1][0, -1, :].float().cpu()  # [d]
                self.samples.append((h, gold_token))

        print(f'Dataset size: {len(self.samples):,} samples')

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        h, token_idx = self.samples[idx]
        token_emb = lm_head_weight[token_idx]  # [d]
        return h, token_emb


train_dataset = PrefixHiddenStateDataset(split='train', max_sentences=5000)
train_loader = DataLoader(train_dataset, batch_size=ROUTER_BATCH_SIZE, shuffle=True, drop_last=True)

### 5b. Router MLP and MNRL Training

In [ ]:
class RouterMLP(nn.Module):
    """
    Lightweight 3-layer MLP that projects hidden state h ∈ R^d
    to a query vector q ∈ R^d_r for cosine similarity search.
    """
    def __init__(self, in_dim: int, hidden_dim: int, out_dim: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.GELU(),
            nn.LayerNorm(hidden_dim),
            nn.Linear(hidden_dim, hidden_dim),
            nn.GELU(),
            nn.LayerNorm(hidden_dim),
            nn.Linear(hidden_dim, out_dim),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return F.normalize(self.net(x), dim=-1)


def mnrl_loss(
    queries: torch.Tensor,    # [B, d_r] normalised query vectors
    keys: torch.Tensor,       # [B, d_r] normalised positive key vectors
    temperature: float = TEMPERATURE,
) -> torch.Tensor:
    """
    Multiple Negatives Ranking Loss (InfoNCE).
    queries[i] should be most similar to keys[i]; all other keys[j!=i] are negatives.
    """
    # keys may live in a different dim; project to same space if needed
    keys_norm = F.normalize(keys, dim=-1)
    sim = (queries @ keys_norm.T) / temperature  # [B, B]
    labels = torch.arange(sim.size(0), device=sim.device)
    return F.cross_entropy(sim, labels)


# Key projector: maps token embedding [d] → [d_r]
# We add a small linear to align token emb dim with router out dim
key_projector = nn.Linear(HIDDEN_DIM, ROUTER_OUT_DIM, bias=False)

router = RouterMLP(HIDDEN_DIM, ROUTER_HIDDEN_DIM, ROUTER_OUT_DIM)
optimizer = torch.optim.AdamW(
    list(router.parameters()) + list(key_projector.parameters()),
    lr=ROUTER_LR,
    weight_decay=1e-2,
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=ROUTER_TRAIN_EPOCHS * len(train_loader)
)

n_params = sum(p.numel() for p in router.parameters())
print(f'Router parameters: {n_params:,}')

In [ ]:
train_losses = []
router.train()
key_projector.train()

for epoch in range(ROUTER_TRAIN_EPOCHS):
    epoch_losses = []
    for hidden_batch, token_emb_batch in tqdm(train_loader, desc=f'Epoch {epoch+1}/{ROUTER_TRAIN_EPOCHS}'):
        # hidden_batch: [B, d],  token_emb_batch: [B, d]
        queries = router(hidden_batch)            # [B, d_r]
        keys = key_projector(token_emb_batch)     # [B, d_r]
        keys = F.normalize(keys, dim=-1)

        loss = mnrl_loss(queries, keys)
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(router.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        epoch_losses.append(loss.item())

    mean_loss = np.mean(epoch_losses)
    train_losses.extend(epoch_losses)
    print(f'Epoch {epoch+1}: mean loss = {mean_loss:.4f}')

# Save checkpoint
torch.save({'router': router.state_dict(), 'key_projector': key_projector.state_dict()},
           RESULTS_DIR / 'router_checkpoint.pt')
print('Router saved.')

In [ ]:
plt.figure(figsize=(8, 4))
# Smooth with running mean
window = max(1, len(train_losses) // 100)
smoothed = np.convolve(train_losses, np.ones(window)/window, mode='valid')
plt.plot(smoothed)
plt.xlabel('Step')
plt.ylabel('MNRL Loss')
plt.title('Dual-Encoder Router Training Loss')
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'router_training_loss.png', dpi=150)
plt.show()

### 5c. Build Inference Index

In [ ]:
# Project all vocabulary embeddings through key_projector → index
router.eval()
key_projector.eval()

print('Building vocabulary index (projecting all token embeddings)...')
with torch.no_grad():
    # Process in chunks to avoid OOM
    chunk_size = 4096
    projected_chunks = []
    for i in range(0, VOCAB_SIZE, chunk_size):
        chunk = lm_head_weight[i:i+chunk_size]  # [chunk, d]
        proj = key_projector(chunk)             # [chunk, d_r]
        projected_chunks.append(F.normalize(proj, dim=-1))
    vocab_index = torch.cat(projected_chunks, dim=0)  # [V, d_r]

print(f'Vocabulary index shape: {vocab_index.shape}')


def get_dual_encoder_shortlist(hidden: torch.Tensor, k: int) -> torch.Tensor:
    """hidden: [d] float32 on CPU."""
    with torch.no_grad():
        q = router(hidden.unsqueeze(0)).squeeze(0)  # [d_r]
    sims = vocab_index @ q                           # [V]
    return sims.topk(k).indices                      # [k]

### 5d. Evaluate Dual-Encoder Router

In [ ]:
dual_enc_results = []
for k in K_VALUES:
    print(f'\n--- Dual-Encoder router K={k} ---')
    r = evaluate_router(get_dual_encoder_shortlist, k=k)
    r['perplexity'] = compute_perplexity(get_dual_encoder_shortlist, k=k)
    r['delta_ppl'] = r['perplexity'] - baseline_ppl
    dual_enc_results.append(r)
    print(f"  Acceptance: {r['acceptance_rate']:.3f}  PPL: {r['perplexity']:.3f}  ΔPPL: {r['delta_ppl']:+.3f}  TPS: {r['tokens_per_sec']:.2f}")

pd.DataFrame(dual_enc_results).to_csv(RESULTS_DIR / 'dual_encoder_results.csv', index=False)

---
## 6. Results and Pareto Charts

In [ ]:
# Assemble all results into a single DataFrame
def label_results(results, name):
    df = pd.DataFrame(results)
    df['router'] = name
    return df

df_all = pd.concat([
    label_results(static_results, 'Static Top-K'),
    label_results(cosine_results, 'Cosine'),
    label_results(cluster_results, 'Cluster'),
    label_results(dual_enc_results, 'Dual-Encoder'),
], ignore_index=True)

# Add baseline row for reference
baseline_row = pd.DataFrame([{
    'k': VOCAB_SIZE,
    'acceptance_rate': 1.0,
    'tokens_per_sec': baseline_summary['tokens_per_sec'],
    'lmhead_frac': baseline_summary['lmhead_frac'],
    'perplexity': baseline_ppl,
    'delta_ppl': 0.0,
    'router': 'Baseline (full vocab)',
}])
df_all = pd.concat([baseline_row, df_all], ignore_index=True)
df_all.to_csv(RESULTS_DIR / 'all_results.csv', index=False)

print(df_all[['router', 'k', 'acceptance_rate', 'delta_ppl', 'tokens_per_sec']].to_string(index=False))

In [ ]:
# ── Plot 1: Acceptance Rate vs. K ────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 5))
routers = ['Static Top-K', 'Cosine', 'Cluster', 'Dual-Encoder']
for name in routers:
    sub = df_all[df_all['router'] == name].sort_values('k')
    ax.plot(sub['k'], sub['acceptance_rate'], marker='o', label=name)

ax.axhline(1.0, color='black', linestyle='--', linewidth=1, label='Baseline (full vocab)')
ax.set_xlabel('Shortlist size K')
ax.set_ylabel('Token acceptance rate')
ax.set_title('Acceptance Rate vs. Shortlist Size K')
ax.set_xscale('log')
ax.legend()
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'acceptance_rate_vs_k.png', dpi=150)
plt.show()

In [ ]:
# ── Plot 2: Perplexity Degradation vs. K ─────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 5))
for name in routers:
    sub = df_all[df_all['router'] == name].sort_values('k')
    ax.plot(sub['k'], sub['delta_ppl'], marker='o', label=name)

ax.axhline(0.0, color='black', linestyle='--', linewidth=1, label='Baseline (ΔPPL=0)')
ax.set_xlabel('Shortlist size K')
ax.set_ylabel('ΔPPL (pruned − full vocab)')
ax.set_title('Perplexity Degradation vs. Shortlist Size K')
ax.set_xscale('log')
ax.legend()
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'ppl_degradation_vs_k.png', dpi=150)
plt.show()

In [ ]:
# ── Plot 3: Tokens/sec vs. K ──────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 5))
for name in routers:
    sub = df_all[df_all['router'] == name].sort_values('k')
    ax.plot(sub['k'], sub['tokens_per_sec'], marker='o', label=name)

ax.axhline(baseline_summary['tokens_per_sec'], color='black', linestyle='--',
           linewidth=1, label='Baseline')
ax.set_xlabel('Shortlist size K')
ax.set_ylabel('Tokens/sec')
ax.set_title('Throughput vs. Shortlist Size K')
ax.set_xscale('log')
ax.legend()
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'throughput_vs_k.png', dpi=150)
plt.show()

In [ ]:
# ── Plot 4: Pareto frontier — Accuracy (1-ΔPPL normalised) vs. Speed ─────────
fig, ax = plt.subplots(figsize=(9, 6))
colours = plt.rcParams['axes.prop_cycle'].by_key()['color']

for i, name in enumerate(routers):
    sub = df_all[df_all['router'] == name].sort_values('k')
    sc = ax.scatter(
        sub['tokens_per_sec'],
        sub['delta_ppl'],
        c=colours[i],
        label=name,
        s=80,
        zorder=3,
    )
    # Annotate K values
    for _, row in sub.iterrows():
        ax.annotate(
            f"K={int(row['k'])/1000:.0f}k",
            (row['tokens_per_sec'], row['delta_ppl']),
            textcoords='offset points', xytext=(5, 5), fontsize=7, color=colours[i]
        )

# Baseline point
ax.scatter(
    baseline_summary['tokens_per_sec'], 0.0,
    marker='*', s=200, color='black', zorder=5, label='Baseline'
)
ax.set_xlabel('Tokens/sec  (higher = better)')
ax.set_ylabel('ΔPPL vs. baseline  (lower = better)')
ax.set_title('Pareto Frontier: Speed vs. Accuracy')
# Arrow indicating desired direction
ax.annotate('', xy=(1.0, 0.0), xytext=(0.85, 0.0),
            xycoords='axes fraction', textcoords='axes fraction',
            arrowprops=dict(arrowstyle='->', color='gray'))
ax.annotate('faster →', xy=(0.87, 0.02), xycoords='axes fraction', color='gray', fontsize=9)
ax.legend(loc='upper right')
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'pareto_frontier.png', dpi=150)
plt.show()

---
## 7. Discussion

### 7.1 Key Findings

*(Populate after running all benchmarks.)*

**Q1 — lm_head fraction:**  
The `lm_head` projection accounts for approximately **X%** of per-token latency on MPS. This confirms the premise that targeting this layer is a viable optimisation direction.

**Q2 — Cosine router as baseline:**  
The cosine router (zero-shot, no training) achieves acceptance rates of **X%** at K=2k, outperforming the static Top-K at equivalent shortlist sizes. This confirms that the hidden state carries meaningful vocabulary locality signal.

**Q3 — Dual-Encoder vs. cosine baseline:**  
At K=2k, the dual-encoder router achieves an acceptance rate of **X%** vs. **Y%** for cosine, with a perplexity degradation of **ΔPPL=Z** vs. the full-vocab baseline. The improvement/gap over cosine is **W%**, validating (or challenging) the training investment.

**Q4 — K sweep:**  
The dual-encoder reaches near-baseline accuracy (ΔPPL < 0.5) at K≈**X**, where the full-vocab size is **V**. The effective compression ratio at this operating point is **V/X ×**.

**Q5 — Cluster router:**  
The cluster-based router is the fastest at inference time (cluster lookup is O(C) not O(V)), but achieves lower acceptance rates than the cosine or dual-encoder at equivalent K.

### 7.2 Blind-Spot Risk

Token acceptance rate < 1.0 means the router occasionally prunes the gold token. Every such event forces the model to pick the best available alternative, propagating an error into the KV cache for all future steps. The perplexity metric captures this cumulative effect: ΔPPL grows non-linearly as K shrinks because early routing errors compound.

This risk is structurally unavoidable for any fixed shortlist size. Practical deployments should set K conservatively (acceptance rate ≥ 0.99) and monitor for domain shift.

### 7.3 Limitations

- Single model (Llama-3.2-1B) and single dataset (WikiText-2).
- The router is trained token-by-token; it does not model the sequential dependency between consecutive routing decisions.
- MPS results may not transfer to CUDA hardware (different memory bandwidth profiles).
- The router overhead (MLP forward + cosine search over K) partially offsets the lm_head savings; FAISS ANN indexing would recover more throughput at large K.

### 7.4 Future Work

1. **Instruction-tuned router (INSTRUCTOR-style):** prepend task instructions to training anchors, enabling the router to adapt its shortlist to different domains or generation styles without retraining.
2. **Composing with speculative decoding / Medusa:** the shortlist router and Medusa heads are orthogonal — combining them should yield additive speedups.
3. **Online adaptation:** use accepted/rejected token signal at inference time to fine-tune the router MLP with online contrastive learning.
4. **FAISS ANN index:** replace brute-force cosine search with an ANN index to decouple router latency from K, enabling larger shortlists at negligible overhead.
5. **CUDA benchmarks:** reproduce on NVIDIA hardware to validate generalisability.

In [ ]:
# Print final summary table
print('\n=== Final Summary ===')
print(f'Baseline perplexity:  {baseline_ppl:.3f}')
print(f'Baseline tokens/sec:  {baseline_summary["tokens_per_sec"]:.2f}')
print(f'lm_head time frac:    {baseline_summary["lmhead_frac"]*100:.1f}%')
print()
summary_cols = ['router', 'k', 'acceptance_rate', 'perplexity', 'delta_ppl', 'tokens_per_sec']
print(df_all[summary_cols].to_string(index=False))